In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
#from sklearn.metrics import (accuracy_score, confusion_matrix, roc_auc_score, f1_score)
from sklearn.metrics import (confusion_matrix, accuracy_score, f1_score, roc_auc_score, recall_score, precision_score)
from sklearn.neighbors import KNeighborsClassifier
from sklearn import neighbors
from imblearn.over_sampling import SMOTE
from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
    StackingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
    BaggingClassifier
)

from sklearn.linear_model import (
    LogisticRegression,
    RidgeClassifier,
    Perceptron,
    SGDClassifier,
    PassiveAggressiveClassifier
)
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN, BorderlineSMOTE
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

In [ ]:
df = df = pd.read_csv('https://raw.githubusercontent.com/muajnstu/Large_Scale_Implementation_of_DSK_Chain/refs/heads/main/filtered%20data/Saimon%20and%20Roni/Clustered_Bank_Data.csv')

X = df.drop(columns=['deposit'])
y = df['deposit']

X = df.drop(columns=['deposit'])
y = df['deposit']

In [ ]:
y_cat = pd.Categorical(y)
y_codes = y_cat.codes

**Applying Random Over Sampling (ros) on whole dataset**

In [ ]:
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)
print("Class distribution after ros:", pd.Series(y_resampled).value_counts())

Class distribution after ros: deposit
0    1856
4    1856
5    1856
1    1856
6    1856
3    1856
7    1856
2    1856
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.3, random_state=46, stratify=y_resampled
)

**Implementation**

In [ ]:
def print_metrics(y_true, y_pred, y_prob=None):
    cm = confusion_matrix(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    num_classes = cm.shape[0]

    if num_classes == 2:
        TN, FP, FN, TP = cm.ravel()
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
        gmean = np.sqrt(specificity * sensitivity)
        type1 = FP / (FP + TN) if (FP + TN) > 0 else 0
        type2 = FN / (TP + FN) if (TP + FN) > 0 else 0
        fmeasure = f1_score(y_true, y_pred, pos_label=1)

        auc = 0
        if y_prob is not None and hasattr(y_prob, "shape") and y_prob.shape[1] > 1:
            try:
                auc = roc_auc_score(y_true, y_prob[:, 1])
            except:
                auc = 0
    else:
        TP = np.diag(cm)
        FP = np.sum(cm, axis=0) - TP
        FN = np.sum(cm, axis=1) - TP
        TN = np.sum(cm) - (FP + FN + TP)

        specificity = np.mean([TN[i] / (TN[i] + FP[i]) if (TN[i] + FP[i]) > 0 else 0 for i in range(num_classes)])
        sensitivity = np.mean([TP[i] / (TP[i] + FN[i]) if (TP[i] + FN[i]) > 0 else 0 for i in range(num_classes)])
        gmean = np.sqrt(specificity * sensitivity)
        type1 = np.mean([FP[i] / (FP[i] + TN[i]) if (FP[i] + TN[i]) > 0 else 0 for i in range(num_classes)])
        type2 = np.mean([FN[i] / (TP[i] + FN[i]) if (TP[i] + FN[i]) > 0 else 0 for i in range(num_classes)])
        fmeasure = f1_score(y_true, y_pred, average='macro')

        auc = 0
        if y_prob is not None and hasattr(y_prob, "shape") and y_prob.shape[1] > 1:
            try:
                auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
            except:
                auc = 0

    return {
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "G-Mean": gmean,
        "Type I Error": type1,
        "Type II Error": type2,
        "F1 Score": fmeasure,
        "AUROC": auc
    }

In [ ]:
def run_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)
    except Exception:
        y_prob = None
    print(f"\nModel: {name}")
    print_metrics(y_test, y_pred, y_prob)

In [ ]:
ml_models = {
    # Original models
    "RandomForest": RandomForestClassifier(random_state=42),
    "ExtraTrees": ExtraTreesClassifier(random_state=42),
    "Bagging": BaggingClassifier(random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=10000, random_state=42, solver='saga'),
    "RidgeClassifier": RidgeClassifier(random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "NaiveBayes": GaussianNB(),
    "Perceptron": Perceptron(random_state=42),
    "SGDClassifier": SGDClassifier(random_state=42),
    #"KNN": KNeighborsClassifier(n_neighbors=3),
    "PassiveAggressive": PassiveAggressiveClassifier(random_state=42),
    #"RBFSVM": SVC(kernel='rbf', probability=True, random_state=42),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(),
    "LightGBM": LGBMClassifier(verbosity=-1, random_state=42),


    "VotingSoft": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42))
        ],
        voting='soft',
        n_jobs=-1
    ),


    "VotingHard": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42))
        ],
        voting='hard',
        n_jobs=-1
    ),


    "VotingWeighted": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False))
        ],
        voting='soft',
        weights=[2, 3, 2, 3, 3],
        n_jobs=-1
    ),


    # Stacking_1:
    "Stacking_LR": StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False))
        ],
        final_estimator=LogisticRegression(max_iter=10000, random_state=42, solver='saga'),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    ),


    # Stacking 2:
    "Stacking_LGBM": StackingClassifier(
        estimators=[
            ('lr', LogisticRegression(max_iter=10000, random_state=42, solver='saga')),
            ('lda', LinearDiscriminantAnalysis()),
            ('qda', QuadraticDiscriminantAnalysis()),
            ('nb', GaussianNB())
        ],
        final_estimator=LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    ),


    # Stacking 3:
    "Stacking_GB": StackingClassifier(
        estimators=[
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('bag', BaggingClassifier(random_state=42)),
            ('sgd', SGDClassifier(random_state=42, loss='modified_huber')),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42))
        ],
        final_estimator=GradientBoostingClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    ),


    # Stacking 4:
    "Stacking_RF": StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('lda', LinearDiscriminantAnalysis()),
            ('nb', GaussianNB()),
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('bag', BaggingClassifier(random_state=42))
        ],
        final_estimator=RandomForestClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    ),
    # Stacking 5:
    "Stacking_ET": StackingClassifier(
        estimators=[
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('lr', LogisticRegression(max_iter=10000, random_state=42, solver='saga')),
            ('lda', LinearDiscriminantAnalysis()),
            ('nb', GaussianNB())
        ],
        final_estimator=ExtraTreesClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    ),
}

In [ ]:
results = []
for name, model in ml_models.items():
    results.append(run_model(name, model, X_train, X_test, y_train, y_test))

df_results = pd.DataFrame(results)
df_results = df_results[["Model","Accuracy","Sensitivity","Specificity","G-Mean","Type I Error","Type II Error","F1 Score","AUROC"]]
df_results = df_results.round(4)
df_results

In [ ]:
df_results = df_results.drop_duplicates(subset="Model").reset_index(drop=True)
df_results

In [ ]:
df_results.to_excel("model_results.xlsx", index=False)
print("Results saved to model_results.xlsx")

Results saved to model_results.xlsx
